# Install packages

In [1]:
!pip install -q \
    numpy==1.26.4 \
    pandas==2.2.3 \
    tqdm==4.67.1 \
    matplotlib==3.9.2 \
    scikit-learn==1.5.2 \
    tensorboard==2.18.0

# PyTorch (Colab / Linux with NVIDIA GPU)
!pip install -q torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# PyTorch (CPU only / macOS)
# !pip install -q torch==2.5.1

# Optional packages
# !pip install -q optuna==4.1.0 gdown==5.2.0



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


# Boss Baseline

You can search the `Boss`, `Strong`, and `Medium` tags to locate the changes from the sample code (Ctrl + F or Command + F).

# **Homework 1: COVID-19 Cases Prediction (Regression)**

Objectives:
* Solve a regression problem with deep neural networks (DNN).
* Understand basic DNN training tips.
* Familiarize yourself with PyTorch.

If you have any questions, please contact the TAs via TA hours, NTU COOL, or email to mlta-2023-spring@googlegroups.com

In [2]:
import shutil
import subprocess
import sys

print(f'Python: {sys.version.split()[0]}')

if shutil.which('nvidia-smi'):
    try:
        print(subprocess.check_output(['nvidia-smi'], text=True))
    except Exception as e:
        print(f'nvidia-smi exists but failed to run: {e}')
else:
    print('nvidia-smi not found. This is fine on CPU / MPS / non-NVIDIA environments.')


Python: 3.10.12
Tue May 12 20:12:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...    On  |   00000000:DC:00.0 Off |                  N/A |
| 30%   30C    P8             21W /  320W |       0MiB /  32760MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------

# Download data
If the Google Drive links below do not work, you can use the dropbox link below or download data from [Kaggle](https://www.kaggle.com/t/a339b77fa5214978bfb8dde62d3151fe), and upload data manually to the workspace.

In [3]:
# # google drive link
# !pip install gdown
# !gdown --id '1BjXalPZxq9mybPKNjF3h5L3NcF7XKTS-' --output covid_train.csv
# !gdown --id '1B55t74Jg2E5FCsKCsUEkPKIuqaY7UIi1' --output covid_test.csv

# # dropbox link
# !wget -O covid_train.csv https://www.dropbox.com/s/lmy1riadzoy0ahw/covid.train.csv?dl=0
# !wget -O covid_test.csv https://www.dropbox.com/s/zalbw42lu4nmhr2/covid.test.csv?dl=0


# Import packages

In [4]:
# Numerical Operations
import math
import random
import numpy as np

# Reading/Writing Data
import pandas as pd
import os
import csv

# For Progress Bar
from tqdm import tqdm

# Pytorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# Matplotlib
import matplotlib.pyplot as plt

# Scikit-learn
from sklearn.feature_selection import SelectKBest, f_regression

# Optuna
try:
    import optuna
except ImportError:
    optuna = None

# For plotting learning curve
from torch.utils.tensorboard import SummaryWriter

from sklearn.model_selection import KFold


# Some Utility Functions

You do not need to modify this part.

In [5]:
def same_seed(seed):
    '''Fixes random number generator seeds for reproducibility.'''
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def train_valid_split(data_set, valid_ratio, seed):
    '''Split provided training data into training set and validation set'''
    valid_set_size = int(valid_ratio * len(data_set))
    train_set_size = len(data_set) - valid_set_size
    train_set, valid_set = random_split(
        data_set,
        [train_set_size, valid_set_size],
        generator=torch.Generator().manual_seed(seed),
    )
    return np.array(train_set), np.array(valid_set)

def predict(test_loader, model, device):
    model.eval() # Set your model to evaluation mode.
    preds = []
    for x in tqdm(test_loader):
        x = x.to(device)
        with torch.no_grad():
            pred = model(x)
            preds.append(pred.detach().cpu())
    preds = torch.cat(preds, dim=0).numpy()
    return preds


# Dataset

In [6]:
class CovidDataset(Dataset):
    '''
    x: Features.
    y: Targets, if none, do prediction.
    '''
    def __init__(self, x, y=None):
        if y is None:
            self.y = y
        else:
            self.y = torch.FloatTensor(y)
        self.x = torch.FloatTensor(x)

    def __getitem__(self, idx):
        if self.y is None:
            return self.x[idx]
        else:
            return self.x[idx], self.y[idx]

    def __len__(self):
        return len(self.x)


# Neural Network Model

Try out different model architectures by modifying the class below. (You could tune config['layer'] to try)

In [7]:
class My_Model(nn.Module):
    def __init__(self, input_dim):
        super(My_Model, self).__init__()
        # TODO: modify model's structure, be aware of dimensions.
        self.layers = nn.Sequential(
            nn.Linear(input_dim, config['layer'][0]),
            nn.ReLU(),
            nn.Linear(config['layer'][0], config['layer'][1]),
            nn.ReLU(),
            nn.Linear(config['layer'][1], 1)
        )

    def forward(self, x):
        x = self.layers(x)
        x = x.squeeze(1) # (B, 1) -> (B)
        return x

# Feature Selection
Choose features you deem useful by modifying the function below.

In [8]:
def select_feat(train_data, valid_data, test_data, no_select_all=True):
    '''Selects useful features to perform regression'''
    global config

    y_train, y_valid = train_data[:, -1], valid_data[:, -1]

    # The first column is the row id, so it is excluded from learning features.
    raw_x_train = train_data[:, 1:-1]
    raw_x_valid = valid_data[:, 1:-1]
    raw_x_test = test_data[:, 1:]

    if not no_select_all:
        feat_idx = list(range(raw_x_train.shape[1]))
    else:
        # (Medium) Rank features with univariate regression scores.
        # (Boss) Tune how many top features to keep.
        k = min(config['k'], raw_x_train.shape[1])
        selector = SelectKBest(score_func=f_regression, k=k)
        selector.fit(raw_x_train, y_train)
        feat_idx = selector.get_support(indices=True).tolist()

    return (
        raw_x_train[:, feat_idx],
        raw_x_valid[:, feat_idx],
        raw_x_test[:, feat_idx],
        y_train,
        y_valid,
    )


# Training Loop

In [9]:
def trainer(train_loader, valid_loader, model, config, device, save_path=None, fold_idx=None, total_folds=None):

    # Define your loss function, do not modify this.
    criterion = nn.MSELoss(reduction='mean')

    # (Strong) Switch between different optimizers.
    # (Strong) Use weight_decay for L2 regularization.
    if config['optim'] == 'SGD':
        if config['no_momentum']:
            optimizer = torch.optim.SGD(
                model.parameters(),
                lr=config['learning_rate'],
                weight_decay=config['weight_decay'],
            )
        else:
            optimizer = torch.optim.SGD(
                model.parameters(),
                lr=config['learning_rate'],
                momentum=config['momentum'],
                weight_decay=config['weight_decay'],
            )
    elif config['optim'] == 'Adam':
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=config['learning_rate'],
            weight_decay=config['weight_decay'],
        )
    else:
        raise ValueError(f"Unsupported optimizer: {config['optim']}")

    # Writer of tensoboard.
    writer = None
    if not config['no_tensorboard']:
        log_dir = './runs' if fold_idx is None else f'./runs/fold_{fold_idx}'
        writer = SummaryWriter(log_dir=log_dir)

    # Create directory of saving models.
    os.makedirs(config['save_dir'], exist_ok=True)

    n_epochs, best_loss, step, early_stop_count = config['n_epochs'], math.inf, 0, 0
    fold_prefix = '' if fold_idx is None else f'[Fold {fold_idx}/{total_folds}] '

    for epoch in range(n_epochs):
        model.train()
        loss_record = []

        # If running on Kaggle, comment out most prints and the train_pbar (use 'for x, y in train_loader' instead), because excessive printing on Kaggle can cause errors.
        # tqdm is a package to visualize your training progress.
        #train_pbar = tqdm(train_loader, position=0, leave=True)
        #for x, y in train_pbar:
        for x, y in train_loader:
            optimizer.zero_grad()               # Set gradient to zero.
            x, y = x.to(device), y.to(device)   # Move your data to device.
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()                     # Compute gradient(backpropagation).

            # (Boss) Gradient clipping for stability.
            if config.get('grad_clip') is not None:
                nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])

            optimizer.step()                    # Update parameters.
            step += 1
            loss_record.append(loss.detach().item())

            # Display current epoch number and loss on tqdm progress bar.
            #train_pbar.set_description(f'Epoch [{epoch+1}/{n_epochs}]')
            #train_pbar.set_postfix({'loss': loss.detach().item()})

        mean_train_loss = sum(loss_record) / len(loss_record)

        model.eval()  # Set your model to evaluation mode.
        loss_record = []
        for x, y in valid_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                pred = model(x)
                loss = criterion(pred, y)
            loss_record.append(loss.item())

        mean_valid_loss = sum(loss_record) / len(loss_record)

        if writer is not None:
            writer.add_scalar('Loss/train', mean_train_loss, step)
            writer.add_scalar('Loss/valid', mean_valid_loss, step)

        if mean_valid_loss < best_loss:
            # Save the best-on-validation model from this fold (used later for fold ensemble).
            best_loss = mean_valid_loss
            if save_path is not None and not config['no_save']:
                torch.save(model.state_dict(), save_path)
            print(f'{fold_prefix}New fold best loss {best_loss:.6f}...')
            early_stop_count = 0
        else:
            early_stop_count += 1

        if early_stop_count >= config['early_stop']:
            print(f'{fold_prefix}Best loss {best_loss:.6f}...')
            print(f'\n{fold_prefix}Model is not improving, so we halt the training session.')
            break

    if writer is not None:
        writer.close()

    return best_loss


# Save predictions

In [10]:
def save_pred(preds, file):
    ''' Save predictions to specified file '''
    with open(file, 'w') as fp:
        writer = csv.writer(fp)
        writer.writerow(['id', 'tested_positive'])
        for i, p in enumerate(preds):
            writer.writerow([i, p])


# Start training!

`config` contains hyper-parameters for training and the path to save your model.

`objective()` is used for automatic parameter tuning, but you could set `AUTO_TUNE_PARAM` to `False` to avoid it.

In [11]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

config = {
    'seed': 5201314,          # Your seed number, you can pick your lucky number. :)
    'k': 18,                  # Number of selected features.
    'layer': [24, 16],        # Hidden dimensions of the MLP.
    'optim': 'Adam',
    'momentum': 0.7,
    'valid_ratio': 0.2,       # validation_size = train_size * valid_ratio
    'n_epochs': 10000,        # Number of epochs.
    'batch_size': 256,
    'learning_rate': 6e-5,
    'weight_decay': 7e-5,
    'grad_clip': 1.0,
    'early_stop': 600,        # If model has not improved for this many consecutive epochs, stop training.
    'save_dir': './models',
    'output_path': './submission.csv',
    'no_select_all': True,    # Whether to skip using all features.
    'no_momentum': True,      # Whether to skip momentum in SGD.
    'no_normal': True,        # Whether to skip normalization.
    'no_k_cross': False,      # Whether to skip fold-wise validation.
    'no_save': False,         # Whether to save model parameters.
    'no_tensorboard': False,  # Whether to write tensorboard logs.
}

# Set the number of folds according to valid_ratio.
n_folds = int(1 / config['valid_ratio'])

# Set seed for reproducibility
same_seed(config['seed'])

training_df = pd.read_csv('./covid_train.csv')
test_df = pd.read_csv('./covid_test.csv')

training_data = training_df.values
raw_test_data = test_df.values

print(f'device: {device}')
print(f'training_data shape: {training_data.shape}')
print(f'raw_test_data shape: {raw_test_data.shape}')
print(f'default config: {config}')

# (Boss) Hyper-parameter search with Optuna.
def objective(trial):
    cfg = config.copy()

    if trial is not None:
        print('\nNew trial here')

        # Define the hyper-parameter search space.
        cfg['optim'] = trial.suggest_categorical('optim', ['SGD', 'Adam'])
        cfg['learning_rate'] = trial.suggest_float('lr', 1e-6, 5e-4, log=True)
        cfg['weight_decay'] = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
        cfg['batch_size'] = trial.suggest_categorical('batch_size', [128, 256])
        cfg['k'] = trial.suggest_int('k_feats', 16, 32)
        cfg['layer'] = [
            trial.suggest_categorical('hidden_dim_1', [16, 24, 32, 48, 64]),
            trial.suggest_categorical('hidden_dim_2', [8, 16, 24, 32]),
        ]

        if cfg['optim'] == 'SGD':
            cfg['no_momentum'] = False
            cfg['momentum'] = trial.suggest_float('momentum', 0.0, 0.9)
        else:
            cfg['no_momentum'] = True

    config.update(cfg)

    # Print the hyper-parameters in use.
    print(
        f'''hyper-parameters:
    optimizer: {cfg['optim']}
    lr: {cfg['learning_rate']}
    batch_size: {cfg['batch_size']}
    weight_decay: {cfg['weight_decay']}
    k: {cfg['k']}
    layer: {cfg['layer']}'''
    )

    fold_test_preds = []
    valid_scores = []
    # (Boss) K-fold cross validation.
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=cfg['seed'])

    for fold, (train_idx, valid_idx) in enumerate(kf.split(training_data), 1):
        print(f'\n===== Fold {fold}/{n_folds} =====')

        train_data = training_data[train_idx].copy()
        valid_data = training_data[valid_idx].copy()
        test_data = raw_test_data.copy()

        if not cfg['no_normal']:
            train_mean = np.mean(train_data[:, 35:-1], axis=0)  # col 0 is id, col 1-34 are state one-hots; the first 35 columns are not normalized.
            train_std = np.std(train_data[:, 35:-1], axis=0)
            train_std[train_std == 0] = 1

            train_data[:, 35:-1] = (train_data[:, 35:-1] - train_mean) / train_std
            valid_data[:, 35:-1] = (valid_data[:, 35:-1] - train_mean) / train_std
            test_data[:, 35:] = (test_data[:, 35:] - train_mean) / train_std

        x_train, x_valid, x_test, y_train, y_valid = select_feat(
            train_data,
            valid_data,
            test_data,
            cfg['no_select_all'],
        )

        print(f'[Fold {fold}/{n_folds}] number of selected features: {x_train.shape[1]}')

        train_dataset = CovidDataset(x_train, y_train)
        valid_dataset = CovidDataset(x_valid, y_valid)
        test_dataset = CovidDataset(x_test)

        pin = (device == 'cuda')
        train_loader = DataLoader(
            train_dataset,
            batch_size=cfg['batch_size'],
            shuffle=True,
            pin_memory=pin,
        )
        valid_loader = DataLoader(
            valid_dataset,
            batch_size=cfg['batch_size'],
            shuffle=False,
            pin_memory=pin,
        )
        test_loader = DataLoader(
            test_dataset,
            batch_size=cfg['batch_size'],
            shuffle=False,
            pin_memory=pin,
        )

        # (Boss) Per-fold seed for diverse initialization.
        same_seed(cfg['seed'] + fold)
        model = My_Model(input_dim=x_train.shape[1]).to(device)
        fold_save_path = os.path.join(cfg['save_dir'], f'model_fold{fold}.ckpt')

        valid_score = trainer(
            train_loader,
            valid_loader,
            model,
            cfg,
            device,
            save_path=fold_save_path,
            fold_idx=fold,
            total_folds=n_folds,
        )
        valid_scores.append(valid_score)
        print(f'[Fold {fold}/{n_folds}] Final best valid loss: {valid_score:.6f}')

        if trial is None or globals().get('SAVE_PER_TRIAL_SUBMISSION', False):
            model.load_state_dict(torch.load(fold_save_path, map_location=device))
            fold_test_preds.append(predict(test_loader, model, device))

        if cfg['no_k_cross']:
            break

    print(f'valid_scores: {valid_scores}')
    print(f'mean_valid_score: {float(np.mean(valid_scores)):.6f}')

    if trial is not None:
        if globals().get('SAVE_PER_TRIAL_SUBMISSION', False) and fold_test_preds:
            trial_preds = np.mean(np.stack(fold_test_preds), axis=0)
            save_pred(trial_preds, f'./submission_trial{trial.number}.csv')
            print(f'Trial {trial.number} submission saved to ./submission_trial{trial.number}.csv')
        return float(np.mean(valid_scores))

    # (Boss) Average test predictions across folds to reduce variance.
    preds = np.mean(np.stack(fold_test_preds), axis=0)
    return preds


AUTO_TUNE_PARAM = False

if AUTO_TUNE_PARAM:
    if optuna is None:
        raise ImportError('Optuna is not installed. Please install it before enabling AUTO_TUNE_PARAM.')

    n_trials = 10  # Number of trials.
    SAVE_PER_TRIAL_SUBMISSION = True  # If True, also write submission_trial{N}.csv for each trial.
    print(f'AUTO_TUNE_PARAM: {AUTO_TUNE_PARAM}\nn_trials: {n_trials}')
    # Hyper-parameter search via Optuna.
    sampler = optuna.samplers.TPESampler(seed=config['seed'])
    study = optuna.create_study(direction='minimize', sampler=sampler)
    study.optimize(objective, n_trials=n_trials)

    # Print the best hyper-parameter set and performance metric.
    print('Best hyperparameters: {}'.format(study.best_params))
    print('Best performance: {:.6f}'.format(study.best_value))

    # Apply the best hyper-parameters and regenerate the submission file.
    best = study.best_params
    config['optim'] = best['optim']
    config['learning_rate'] = best['lr']
    config['weight_decay'] = best['weight_decay']
    config['batch_size'] = best['batch_size']
    config['k'] = best['k_feats']
    config['layer'] = [best['hidden_dim_1'], best['hidden_dim_2']]
    if best['optim'] == 'SGD':
        config['no_momentum'] = False
        config['momentum'] = best['momentum']
    else:
        config['no_momentum'] = True
    preds = objective(None)
    save_pred(preds, config['output_path'])
    print(f'Submission saved to: {config["output_path"]}')
else:
    print(f'AUTO_TUNE_PARAM: {AUTO_TUNE_PARAM}')
    preds = objective(None)
    save_pred(preds, config['output_path'])
    print(f'Submission saved to: {config["output_path"]}')


device: cuda
training_data shape: (3009, 89)
raw_test_data shape: (997, 88)
default config: {'seed': 5201314, 'k': 18, 'layer': [24, 16], 'optim': 'Adam', 'momentum': 0.7, 'valid_ratio': 0.2, 'n_epochs': 10000, 'batch_size': 256, 'learning_rate': 6e-05, 'weight_decay': 7e-05, 'grad_clip': 1.0, 'early_stop': 600, 'save_dir': './models', 'output_path': './submission.csv', 'no_select_all': True, 'no_momentum': True, 'no_normal': True, 'no_k_cross': False, 'no_save': False, 'no_tensorboard': False}
AUTO_TUNE_PARAM: False
hyper-parameters:
    optimizer: Adam
    lr: 6e-05
    batch_size: 256
    weight_decay: 7e-05
    k: 18
    layer: [24, 16]

===== Fold 1/5 =====
[Fold 1/5] number of selected features: 18
[Fold 1/5] New fold best loss 414.961029...
[Fold 1/5] New fold best loss 407.326742...
[Fold 1/5] New fold best loss 399.781494...
[Fold 1/5] New fold best loss 392.328827...
[Fold 1/5] New fold best loss 384.957631...
[Fold 1/5] New fold best loss 377.673258...
[Fold 1/5] New fold be

/tmp/ipykernel_14726/3545322648.py:153: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(fold_save_path, map_location=device))


[Fold 1/5] Best loss 0.710600...

[Fold 1/5] Model is not improving, so we halt the training session.
[Fold 1/5] Final best valid loss: 0.710600


100%|██████████| 4/4 [00:00<00:00, 950.77it/s]


===== Fold 2/5 =====
[Fold 2/5] number of selected features: 18
[Fold 2/5] New fold best loss 480.903737...
[Fold 2/5] New fold best loss 476.964152...
[Fold 2/5] New fold best loss 472.872889...
[Fold 2/5] New fold best loss 468.478327...


...

[Fold 5/5] New fold best loss 354.116002...
[Fold 5/5] New fold best loss 346.171483...
[Fold 5/5] New fold best loss 338.030825...
[Fold 5/5] New fold best loss 329.986366...
[Fold 5/5] New fold best loss 322.105840...
[Fold 5/5] New fold best loss 314.417323...
[Fold 5/5] New fold best loss 306.883415...
[Fold 5/5] New fold best loss 299.456355...
[Fold 5/5] New fold best loss 292.107061...
[Fold 5/5] New fold best loss 284.848969...
[Fold 5/5] New fold best loss 277.673960...
[Fold 5/5] New fold best loss 270.584239...
[Fold 5/5] New fold best loss 263.571757...
[Fold 5/5] New fold best loss 256.624522...
[Fold 5/5] New fold best loss 249.722417...
[Fold 5/5] New fold best loss 242.841530...
[Fold 5/5] New fold best loss 235.981766...
[Fold 5/5] New fold best loss 229.145213...
[Fold 5/5] New fold best loss 222.340960...
[Fold 5/5] New fold best loss 215.576693...
[Fold 5/5] New fold best loss 208.851901...
[Fold 5/5] New fold best loss 202.181285...
[Fold 5/5] New fold best loss 19

/tmp/ipykernel_14726/3545322648.py:153: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(fold_save_path, map_location=device))


[Fold 5/5] Best loss 0.844883...

[Fold 5/5] Model is not improving, so we halt the training session.
[Fold 5/5] Final best valid loss: 0.844883


100%|██████████| 4/4 [00:00<00:00, 907.71it/s]

valid_scores: [0.7105997006098429, 0.8122193813323975, 0.8875049750010172, 0.989913801352183, 0.8448830445607504]
mean_valid_score: 0.849024
Submission saved to: ./submission.csv


# Plot learning curves with `tensorboard` (optional)

`tensorboard` is a tool that allows you to visualize your training progress.

If this block does not display your learning curve, please wait for a few minutes and re-run it. It might take some time to load your logging information.

In [12]:
%reload_ext tensorboard
%tensorboard --logdir=./runs/


Reusing TensorBoard on port 6006 (pid 12676), started 29 days, 20:12:41 ago. (Use '!kill 12676' to kill it.)